<img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:100px" alt="MCUNet Logo">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Déploiement TinyML avec MCUNet </h1></center>
<center><h2> Notebook 02 : Inférence en Python (INT8) </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

>
> **Objectif :** Charger un modèle TFLite quantifié, comprendre les paramètres de quantification, et réaliser une inférence correcte en gérant l'échelle (`scale`) et le point zéro (`zero_point`).
>
> **Durée estimée :** 45 minutes
>
> **Prérequis :** Compréhension des types de données (float32 vs int8).
>


### Concepts clés
>
> 1. **Quantification INT8** : Processus convertissant les poids et les activations flottants (float32) en entiers 8-bits. Cela divise la taille du modèle par 4 et accélère l'inférence sur MCU.
> 2. **Équation de quantification** : `valeur_réelle = scale * (valeur_quantifiée - zero_point)`.
> 3. **TFLite Interpreter** : Le moteur d'exécution de TensorFlow Lite qui permet d'allouer les tenseurs et d'invoquer l'inférence.


--- 
### 1. Chargement du modèle sélectionné

Dans le notebook précédent, nous avons déterminé que `mcunet-vww1` était le meilleur modèle pour notre STM32F746. Chargeons son fichier `.tflite`.

In [ ]:
import os
import numpy as np
import tflite_runtime.interpreter as tflite
import matplotlib.pyplot as plt
from PIL import Image

# Le modèle a été téléchargé par le Dockerfile dans le cache
model_path = os.path.expanduser("~/.mcunet/mcunet-vww1.tflite")

print(f"Chargement du modèle : {model_path}")

# Initialiser l'interpréteur TFLite
interpreter = tflite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

print("Modèle alloué avec succès.")

--- 
### 2. Inspection des tenseurs d'entrée/sortie

Pour utiliser un modèle quantifié, il est **indispensable** de récupérer ses paramètres de quantification. Sinon, vous ne saurez pas comment préparer votre image d'entrée.

In [ ]:
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print("--- Détails de l'entrée ---")
print(f"Shape attendue : {input_details['shape']}")
print(f"Type de donnée : {input_details['dtype']}")
print(f"Paramètres de quantification : {input_details['quantization']}")

print("\n--- Détails de la sortie ---")
print(f"Shape attendue : {output_details['shape']}")
print(f"Type de donnée : {output_details['dtype']}")
print(f"Paramètres de quantification : {output_details['quantization']}")

input_scale, input_zero_point = input_details['quantization']
input_size = input_details['shape'][1:3] # (height, width)

--- 
### 3. Préparation (Preprocessing) d'une image

Prenons une image de notre dataset. Pour qu'elle soit acceptée par le modèle, il faut :
1. La redimensionner à la bonne taille.
2. La normaliser dans la plage d'origine (généralement [-1, 1] ou [0, 1] selon l'entraînement).
3. La **quantifier** en `int8` (car l'entrée attendue est `int8`).

In [ ]:
def get_sample_image(is_person=True):
    base_dir = "/dataset/vww_minival"
    target_dir = os.path.join(base_dir, "person" if is_person else "non_person")
    files = os.listdir(target_dir)
    return os.path.join(target_dir, files[0]) # On prend la première

img_path = get_sample_image(is_person=True)
img = Image.open(img_path).convert('RGB')

# 1. Redimensionner
img_resized = img.resize((input_size[1], input_size[0]))
input_data = np.expand_dims(img_resized, axis=0)

# 2. Normalisation float (MCUNet utilise [0, 1] puis normalisation ImageNet, 
# mais pour simplifier ici on utilise une normalisation standard [-1, 1])
input_float = (np.float32(input_data) - 127.5) / 127.5

print(f"Plage float: [{np.min(input_float):.2f}, {np.max(input_float):.2f}]")

#### L'Exercice Piège

Essayons de quantifier les données en utilisant un simple cast, ce que font 90% des débutants.

In [ ]:
# MAUVAIS PREPROCESSING
bad_input_int8 = input_float.astype(np.int8)
print("--- MAUVAISE QUANTIFICATION ---")
print(f"Valeurs: {bad_input_int8[0, 0, 0]}")
print(f"Unique values count: {len(np.unique(bad_input_int8))}")

Toutes les valeurs ont été écrasées en 0 ou -1. L'information de l'image est détruite !

### À toi de jouer !

Implémente la quantification correcte. Rappel de la formule :
`valeur_quantifiée = (valeur_réelle / scale) + zero_point`

In [ ]:
# BON PREPROCESSING

# TODO: Calcule l'entrée quantifiée correctement avec input_scale et input_zero_point
# N'oublie pas d'arrondir (np.round) et de caster en np.int8 à la fin.
# good_input_int8 = ...

# Remplace cette ligne par ton code :
good_input_int8 = np.zeros_like(input_float, dtype=np.int8)

print("--- BONNE QUANTIFICATION ---")
print(f"Valeurs (échantillon): {good_input_int8[0, 0, :5]}")
print(f"Unique values count: {len(np.unique(good_input_int8))}")

--- 
### 4. Exécution de l'Inférence

Maintenant que notre tenseur est bien préparé, invoquons le modèle.

In [ ]:
# On place les données d'entrée dans le modèle
interpreter.set_tensor(input_details['index'], good_input_int8)

# Exécution de l'inférence
interpreter.invoke()

# Récupération du résultat
output_data = interpreter.get_tensor(output_details['index'])
print(f"Sortie brute (int8) : {output_data}")

Cette sortie est un entier quantifié. Nous devons la reconvertir en probabilité (float32) pour la lire, en utilisant la même formule, mais à l'envers :

In [ ]:
output_scale, output_zero_point = output_details['quantization']

# TODO: Dé-quantifie la sortie pour obtenir un tableau de probabilités en float32
# probas_float = ...

# Remplace cette ligne :
probas_float = np.array([[0.0, 0.0]])

print(f"Probabilités : {probas_float}")
classe_predite = np.argmax(probas_float[0])
classes = ["Non-personne", "Personne"]
print(f"\nPrédiction finale : {classes[classe_predite]} ({probas_float[0][classe_predite]*100:.1f}%)")

plt.imshow(img)
plt.title(f"Prédiction: {classes[classe_predite]}")
plt.axis('off')
plt.show()

--- 
### Points d'attention

<div class="alert alert-danger">
<i class='fa fa-exclamation-circle'></i> &emsp; 
  <strong>L'erreur n°1 en déploiement TFLite Int8 :</strong> oublier la conversion avec le <code>scale</code> et le <code>zero_point</code>. Ne faites <strong>JAMAIS</strong> de simple <code>img.astype(np.int8)</code>. Le réseau produira des résultats qui ont l'air aléatoires et vous penserez que le modèle est cassé, alors que c'est seulement votre prétraitement qui est mauvais.
</div>

--- 
### Questions de réflexion

1. Que se passerait-il si le modèle était entraîné avec une normalisation des images entre `[0, 1]` mais que vous faisiez une inférence en fournissant des images normalisées entre `[-1, 1]` (avant la quantification) ? Comment le modèle réagirait-il ?
2. Pourquoi les valeurs de `scale` et `zero_point` sont-elles différentes pour l'entrée et pour la sortie ?